<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">

# Oanda Master Class

## First Steps

Dr Yves J Hilpisch | The Python Quants | The AI Machine

<td><a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://aimachine.io" target="_blank">http://aimachine.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a></td>

## Imports

In [1]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


Cloning into 'python_for_algo_trading_addon'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 56 (delta 22), reused 56 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 680.56 KiB | 1.71 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [5]:
!pip install git+https://github.com/yhilpisch/tpqoa


  Cloning https://github.com/yhilpisch/tpqoa to /tmp/pip-req-build-48o4ahw8
  Running command git clone --filter=blob:none --quiet https://github.com/yhilpisch/tpqoa /tmp/pip-req-build-48o4ahw8
  Resolved https://github.com/yhilpisch/tpqoa to commit 4a79f30b7095642844ef99741d9bfcb952e394db
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.7 MB/s eta 0:00:00
  Created wheel for tpqoa: filename=tpqoa-0.0.56-py3-none-any.whl size=11595 sha256=d7156ca00aa782fb9e4a6040054c641cf7e64788bba2d8616cf7f9c0cc7c2db9
  Stored in directory: /tmp/pip-ephem-wheel-cache-47afyzh4/wheels/57/7c/ca/f178e2b2385c72fdf70b681849f6d11c16cdd9a38c428e4920
  Created wheel for v20: filename=v20-3.0.25.0-py3-none-any.whl size=85779 sha256=fee90733b7c410083612abeaa1241ba074ff1c256bbbd59e4d54f45bb5501ac5
  Stored in directory: /root/.cache/pip/wheels/

In [6]:
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')

## API Connection

In [7]:
import tpqoa

In [8]:
api = tpqoa.tpqoa('/content/python_for_algo_trading_addon/pyalgo.cfg')

KeyError: 'oanda'

## Available Instruments

In [ ]:
ins = api.get_instruments()

In [ ]:
ins[:7]

In [ ]:
len(ins)

## Historical Data

In [ ]:
help(api.get_history)

In [ ]:
instrument = 'EUR_USD'
# instrument = 'DE30_EUR'
# instrument = 'BCO_USD'
start = '2019-01-01'
end = '2020-03-13'
granularity = 'D'
price = 'M'

In [ ]:
data = api.get_history(instrument, start, end, granularity, price, localize=True)

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data['c'].plot(figsize=(10, 6));

In [ ]:
instrument = 'EUR_USD'
# instrument = 'DE30_EUR'
# instrument = 'BCO_USD'
start = '2020-03-01'
end = '2020-03-13'
granularity = 'M5'
price = 'M'

In [ ]:
%%time
data = api.get_history(instrument, start, end,
                       granularity, price, localize=True)

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data['c'].plot(figsize=(10, 6));

## Momentum Strategy

In [ ]:
instrument = 'EUR_USD'
start = '2020-03-12'
end = '2020-03-13'
granularity = 'M1'
price = 'M'

In [ ]:
%%time
data = api.get_history(instrument, start, end,
                       granularity, price, localize=True)

In [ ]:
data.info()

In [ ]:
data['c'].plot(figsize=(10, 6));

In [ ]:
data['r'] = np.log(data['c'] / data['c'].shift(1))

In [ ]:
data['m'] = data['r'].rolling(6).mean().shift(1)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data['p'] = np.where(data['m'] > 0, 1, -1)

In [ ]:
data['s'] = data['p'] * data['r']

In [ ]:
data.head(8)

In [ ]:
data[['r', 's']].sum().apply(np.exp)  # gross performance

In [ ]:
data[['r', 's']].cumsum().apply(np.exp).plot(figsize=(10, 6));  # gross performance

In [ ]:
ax = data[['r', 's']].cumsum().apply(np.exp).plot(figsize=(10, 6))
data['p'].plot(ax=ax, secondary_y='p', alpha=0.5);

In [ ]:
sum(data['p'].diff() != 0)

In [ ]:
pips = 0.00012
ptc = pips / data['c'].mean()
ptc

In [ ]:
data['s_tc'] = np.where(data['p'].diff() != 0, data['s'] - ptc, data['s'])

In [ ]:
data[['r', 's', 's_tc']].sum().apply(np.exp)  # gross performance

In [ ]:
data[['r', 's', 's_tc']].cumsum().apply(np.exp).plot(figsize=(10, 6));  # gross performance

## Streaming Data

In [ ]:
api.stream_data('BTC_USD', stop=10)

In [ ]:
api.stream_data('EUR_USD', stop=10)

## Placing Orders

In [ ]:
api.create_order('EUR_USD', units=10000)

In [ ]:
api.create_order('EUR_USD', units=-10000)

In [ ]:
from pprint import pprint

In [ ]:
order = api.create_order('EUR_USD', units=10000, suppress=True, ret=True)

In [ ]:
pprint(order)

In [ ]:
order = api.create_order('EUR_USD', units=-10000, suppress=True, ret=True)

In [ ]:
pprint(order)

In [ ]:
order = api.create_order('EUR_USD', units=-10000,
                         sl_distance=0.0025,
                         suppress=True, ret=True)

In [ ]:
order = api.create_order('EUR_USD', units=10000,
                         suppress=True, ret=True)

## Recent Transactions

In [ ]:
order['id']

In [ ]:
api.get_transaction(tid=int(order['id']) - 2)

In [ ]:
api.print_transactions(tid=int(order['id']) - 12)

## Account Summary

In [ ]:
api.get_account_summary()

<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">